# 03 — Evaluate a saved training run

**Goal:** inspect what changed after training.

This notebook reads the output folder created by `train.py`, displays the saved result figures, and compares a **randomly initialized CNN** with the **trained CNN**.

It compares the two models using the same ideas introduced in Notebook 2: logits, softmax probabilities, the ideal one-hot target for the true class, and cross-entropy loss.


## 1. Select a run folder

By default, this notebook loads the latest run recorded in `outputs/latest_run.txt`.

To inspect an older run, manually set `RUN_DIR` to a specific run folder.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from IPython.display import Image, display
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from model import build_model

# Set this to a specific run folder if you do not want the latest run.
RUN_DIR = None

if RUN_DIR is None:
    latest_path = PROJECT_ROOT / "outputs" / "latest_run.txt"
    if not latest_path.exists():
        raise FileNotFoundError(
            "outputs/latest_run.txt not found. Run training first:\n"
            "python train.py --config config.yaml"
        )
    RUN_DIR = Path(latest_path.read_text().strip())
    if not RUN_DIR.is_absolute():
        RUN_DIR = PROJECT_ROOT / RUN_DIR

RUN_DIR = Path(RUN_DIR)
print("Using run folder:", RUN_DIR)
if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run folder does not exist: {RUN_DIR}")

## 2. Load the saved config and metrics

Every run saves a copy of the config as `config_used.yaml`. This makes the result reproducible.


In [ ]:
config_path = RUN_DIR / "config_used.yaml"
with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Config used for this run:")
print(yaml.safe_dump(config, sort_keys=False))


In [ ]:
metrics_path = RUN_DIR / "logs" / "metrics.csv"
metrics = pd.read_csv(metrics_path)
display(metrics)

last_row = metrics.iloc[-1]
print(f"Final validation accuracy: {last_row['val_accuracy']:.3f}")
print(f"Final validation loss:     {last_row['val_loss']:.4f}")


## 3. Show the saved training curves

The training script saves the loss and accuracy curves automatically.

A successful short workshop run should usually show:

```text
training loss decreases
validation accuracy is better than random guessing
```

For four classes, random guessing is about 25% accuracy.


In [ ]:
fig_path = RUN_DIR / "figures" / "loss_accuracy_curve.png"
display(Image(filename=str(fig_path)))


## 4. Load the trained model and a random model

We compare two models with the **same architecture**:

1. `random_model`: newly initialized CNN that has not learned from data.
2. `trained_model`: CNN loaded from the saved checkpoint.

This helps answer: **what changed after training?**


In [ ]:
device = torch.device("cpu")  # Keep notebook evaluation portable.

checkpoint_path = RUN_DIR / "checkpoints" / "best_model.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)

class_names = checkpoint["class_names"]
num_classes = len(class_names)

random_model = build_model(config, num_classes=num_classes).to(device)
trained_model = build_model(config, num_classes=num_classes).to(device)
trained_model.load_state_dict(checkpoint["model_state_dict"])

random_model.eval()
trained_model.eval()

print("Classes:", class_names)
print("Best validation accuracy saved in checkpoint:", checkpoint.get("val_accuracy"))


## 5. Build a validation batch

This notebook rebuilds the validation dataset directly using `ImageFolder`, so the image-label folder structure remains visible.


In [ ]:
image_size = int(config["data"]["image_size"])
data_dir = PROJECT_ROOT / config["data"]["data_dir"]
val_dir = data_dir / "val"

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

val_dataset = datasets.ImageFolder(val_dir, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)
images, labels = next(iter(val_loader))
images = images.to(device)
labels = labels.to(device)

print("Validation batch shape:", tuple(images.shape))
print("Label shape:", tuple(labels.shape))


## 6. Random model vs trained model: logits, probabilities, and loss

Notebook 2 explained the prediction pipeline for one model:

```text
image → CNN → logits → softmax probabilities → predicted class
```

Here we use the same idea to compare two models on the same validation image:

- a randomly initialized model
- the trained model loaded from the saved checkpoint

The useful question is not only which class has the highest probability, but also: **how much probability did the model assign to the true class?**


In [ ]:
with torch.no_grad():
    random_logits = random_model(images)
    trained_logits = trained_model(images)

    random_probs = F.softmax(random_logits, dim=1)
    trained_probs = F.softmax(trained_logits, dim=1)

    random_preds = random_probs.argmax(dim=1)
    trained_preds = trained_probs.argmax(dim=1)

print("Random logits shape:", tuple(random_logits.shape))
print("Trained logits shape:", tuple(trained_logits.shape))
print("Random batch accuracy:", float((random_preds == labels).float().mean()))
print("Trained batch accuracy:", float((trained_preds == labels).float().mean()))


### One-image comparison: logits, softmax, and one-image cross-entropy loss

For one image, the cross-entropy loss can be understood as:

```text
loss = -log(softmax probability assigned to the true class)
```

The probability plot shows two things at the true class:

- an **orange dotted box** reaching probability **1** (the ideal one-hot target)
- an **orange star** at the model probability for the true class, call it $p_t$

This helps you compare the ideal target with the model output. The loss is **not** a simple vertical difference; it is computed as $-\log(p_t)$.

In training code we still pass **raw logits** into `nn.CrossEntropyLoss`, because PyTorch performs the stable log-softmax calculation internally.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

def show_image_tensor(ax, image_tensor, title=None):
    image_np = image_tensor.detach().cpu().permute(1, 2, 0).numpy()
    image_np = np.clip(image_np, 0, 1)
    ax.imshow(image_np)
    if title:
        ax.set_title(title)
    ax.axis("off")

def plot_logits_probability_loss_comparison(example_idx=0):
    criterion_none = torch.nn.CrossEntropyLoss(reduction="none")

    true_idx = int(labels[example_idx])
    true_name = class_names[true_idx]

    rows = [
        ("Random model", random_logits, random_probs, random_preds),
        ("Trained model", trained_logits, trained_probs, trained_preds),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(14, 7))

    for row_idx, (model_name, logits_all, probs_all, preds_all) in enumerate(rows):
        one_logits = logits_all[example_idx:example_idx + 1]
        one_label = labels[example_idx:example_idx + 1]
        loss_value = float(criterion_none(one_logits, one_label)[0])

        logits_np = one_logits[0].detach().cpu().numpy()
        probs_np = probs_all[example_idx].detach().cpu().numpy()
        pred_name = class_names[int(preds_all[example_idx])]
        target_prob = float(probs_np[true_idx])

        show_image_tensor(axes[row_idx, 0], images[example_idx], title=f"{model_name}\ntrue label: {true_name}")

        axes[row_idx, 1].bar(class_names, logits_np)
        axes[row_idx, 1].axhline(0, linewidth=1)
        axes[row_idx, 1].set_title(f"Raw logits\npred: {pred_name}")
        axes[row_idx, 1].set_ylabel("Logit value")
        axes[row_idx, 1].tick_params(axis="x", rotation=25)

        x = np.arange(len(class_names))
        bar_width = 0.8
        axes[row_idx, 2].bar(x, probs_np, alpha=0.85)
        target_left = true_idx - bar_width / 2
        target_box = Rectangle((target_left, 0), bar_width, 1.0, fill=False, edgecolor='tab:orange', linewidth=2.0, linestyle=':')
        axes[row_idx, 2].add_patch(target_box)
        axes[row_idx, 2].scatter([true_idx], [target_prob], marker='*', s=220, color='tab:orange', zorder=3)
        axes[row_idx, 2].set_xticks(x)
        axes[row_idx, 2].set_xticklabels(class_names, rotation=25, ha='right')
        axes[row_idx, 2].set_ylim(0, 1.05)
        axes[row_idx, 2].set_title(f"Softmax probabilities\nloss = -log({target_prob:.3f}) = {loss_value:.3f}")
        axes[row_idx, 2].set_ylabel("Probability")
        legend_handles = [
            Rectangle((0, 0), 1, 1, facecolor=plt.rcParams['axes.prop_cycle'].by_key()['color'][0], alpha=0.85, label='Softmax probability'),
            Rectangle((0, 0), 1, 1, fill=False, edgecolor='tab:orange', linewidth=2.0, linestyle=':', label='Ideal target for true class = 1'),
            Line2D([0], [0], marker='*', color='tab:orange', linestyle='None', markersize=12, label=f'Model probability for true class = {target_prob:.3f}'),
        ]
        axes[row_idx, 2].legend(handles=legend_handles, loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()

plot_logits_probability_loss_comparison(example_idx=0)


## 7. Prediction grid: random model vs trained model

This grid uses the same images for both models.

For each image, compare:

```text
true label
random model prediction
trained model prediction
trained model confidence
```


In [ ]:
def plot_random_vs_trained_grid(images, labels, random_preds, trained_preds, trained_probs, class_names, n=8):
    n = min(n, images.shape[0])
    fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
    axes = axes.reshape(-1)

    for i, ax in enumerate(axes[:n]):
        show_image_tensor(ax, images[i])
        true_name = class_names[int(labels[i])]
        rand_name = class_names[int(random_preds[i])]
        train_name = class_names[int(trained_preds[i])]
        conf = float(trained_probs[i].max())
        correct_marker = "✓" if trained_preds[i] == labels[i] else "✗"
        ax.set_title(
            f"true: {true_name}\nrandom: {rand_name}\ntrained: {train_name} {correct_marker}\nconf: {conf:.2f}",
            fontsize=9,
        )

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

plot_random_vs_trained_grid(images, labels, random_preds, trained_preds, trained_probs, class_names)

## 8. First-layer feature maps: random model vs trained model

The first convolution layer produces multiple feature maps.

Feature maps are not final predictions. They are intermediate activations. After training, they may become more structured, but they are not always easy to interpret visually.


In [ ]:
def first_conv_relu_activations(model, image_batch):
    # SmallCNN.features = Conv2d, ReLU, MaxPool2d, Conv2d, ReLU, MaxPool2d
    with torch.no_grad():
        x = model.features[0](image_batch)
        x = model.features[1](x)
    return x

sample = images[0:1]
random_maps = first_conv_relu_activations(random_model, sample)
trained_maps = first_conv_relu_activations(trained_model, sample)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for row, (title, maps) in enumerate([("Random model", random_maps), ("Trained model", trained_maps)]):
    axes[row, 0].imshow(sample[0].permute(1, 2, 0).detach().cpu().numpy())
    axes[row, 0].set_title(f"{title}\ninput")
    axes[row, 0].axis("off")

    for col in range(1, 5):
        axes[row, col].imshow(maps[0, col - 1].detach().cpu().numpy(), cmap="gray")
        axes[row, col].set_title(f"map {col - 1}")
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 9. Saved confusion matrix and prediction grid

The training script also saved full-run evaluation figures. These are the figures students can submit.


In [ ]:
confusion_path = RUN_DIR / "figures" / "confusion_matrix.png"
display(Image(filename=str(confusion_path)))


In [ ]:
prediction_path = RUN_DIR / "figures" / "prediction_grid.png"
display(Image(filename=str(prediction_path)))


## 10. Interpretation questions

Answer these in 2–3 sentences for the final submission.

1. Did training loss decrease?
2. Did validation accuracy improve?
3. Compared with the random model, what changed in the trained model's predictions?
4. Does the trained model assign higher probability to reasonable classes?
6. Which classes are most often confused?
7. What hyperparameter did you change in `config.yaml`, and what happened?

Example answer:

```text
I changed the learning rate from 0.001 to 0.01. The model learned faster at first, but the validation accuracy became less stable. Compared with the random model, the trained model assigned higher probability to the correct class on many validation images.
```
